<a href="https://colab.research.google.com/github/harshit-nitt/Customer-Segmentation/blob/main/Physics_Informed_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Importing Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import mean_squared_error

RNG = np.random.default_rng(42)
G = 9.81


##Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style for clean aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 10, 'axes.labelsize': 12, 'axes.titlesize': 14})


# 1. LOAD AND INSPECT THE DATASET


df = pd.read_csv("/content/DATASET_PIMP.csv")
print(f"Successfully loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns.\n")


print("--- Dataset Information Summary ---")
print(df.info())

print("\n--- Summary Statistics ---")
print(df.describe().T.round(2))

print("\n--- Checking for Missing Values ---")
print(df.isnull().sum())


# 2. DEMOGRAPHICS & ANTHROPOMETRICS EDA
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sex distribution
sns.countplot(data=df, x='sex', hue='sex', palette='pastel', legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Distribution of Subject Sex")

# Age distribution
sns.histplot(data=df, x='age', kde=True, color='skyblue', ax=axes[0, 1])
axes[0, 1].set_title("Age Distribution of Cohort")

# Height vs Weight stratified by Sex
sns.scatterplot(data=df, x='height_cm', y='weight_kg', hue='sex', palette='Set1', alpha=0.8, ax=axes[1, 0])
axes[1, 0].set_title("Height vs. Weight (Anthropometric Validity)")

# BMI vs Leg Stiffness
sns.scatterplot(data=df, x='bmi', y='k_leg', hue='sex', palette='Set1', alpha=0.8, ax=axes[1, 1])
axes[1, 1].set_title("BMI vs. Derived Leg Stiffness (k_leg)")

plt.tight_layout()
plt.suptitle("Demographics & Physical Characteristics", y=1.02, fontsize=16, fontweight='bold')
plt.show()

# =============================================================================
# 3. GAIT PARAMETERS & BIOMECHANICAL RELATIONSHIPS
# =============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution of experimental walking speeds
sns.countplot(data=df, x='speed_kmph', hue='speed_kmph', palette='viridis', legend=False, ax=axes[0])
axes[0].set_title("Samples per Walking Speed")
axes[0].set_xlabel("Speed (km/h)")

# Froude number vs Speed (Should be quadratic but shifted by leg length variances)
sns.scatterplot(data=df, x='speed_kmph', y='Fr', hue='height_cm', palette='magma', ax=axes[1])
axes[1].set_title("Froude Number vs. Speed\n(Color-coded by Height)")
axes[1].set_ylabel("Froude Number (Fr)")

# Froude number vs Body Weight Normalised Peak Force
df['F_meas_peak_BW'] = df['F_measured_peak'] / df['W']
sns.regplot(data=df, x='Fr', y='F_meas_peak_BW', scatter_kws={'alpha':0.5}, line_kws={'color':'red'}, ax=axes[2])
axes[2].set_title("Peak Force (Multiples of Body Weight) vs. Fr")
axes[2].set_ylabel("Peak Measured Force / BW")

plt.tight_layout()
plt.show()


# 4. CORRELATION ANALYSIS (Features vs. Fourier Parameters)

plt.figure(figsize=(12, 10))
# Select key numerical variables for a correlation matrix
corr_cols = ['height_cm', 'weight_kg', 'age', 'speed_kmph', 'bmi', 'Fr', 'k_leg', 'zeta',
             'b1_meas', 'b2_meas', 'b3_meas', 'b4_meas', 'F_measured_peak']
corr_matrix = df[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, square=True, annot_kws={"size": 8})
plt.title("Correlation Matrix: Body Traits, Biomechanics, & Measured Harmonics", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


# 5. ANALYSIS OF PHYSICS RESIDUALS (What the ML needs to learn)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

# Distribution of residuals
for i in range(1, 5):
    sns.histplot(data=df, x=f'res_b{i}', kde=True, color='crimson', ax=axes[i-1], alpha=0.6)
    axes[i-1].axvline(0, color='black', linestyle='--', linewidth=1.5)
    axes[i-1].set_title(f"Residual Distribution for b{i} (Measured - Physics)")
    axes[i-1].set_xlabel(f"Residual b{i} (N)")

plt.tight_layout()
plt.suptitle("Physics Model Discrepancy (Residuals)", y=1.02, fontsize=16, fontweight='bold')
plt.show()

# Investigating the Soft-Tissue BMI Unmodeled Effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# b2 residual vs BMI
sns.regplot(data=df, x='bmi', y='res_b2', color='purple', scatter_kws={'alpha':0.5}, ax=axes[0])
axes[0].set_title("b2 Residual vs. BMI\n(Soft-Tissue Asymmetry Cushioning)")
axes[0].set_ylabel("b2 Residual (N)")

# b3 residual vs BMI
sns.regplot(data=df, x='bmi', y='res_b3', color='teal', scatter_kws={'alpha':0.5}, ax=axes[1])
axes[1].set_title("b3 Residual vs. BMI\n(Soft-Tissue Dip Dampening)")
axes[1].set_ylabel("b3 Residual (N)")

plt.tight_layout()
plt.show()


# 6. COMPARING COMPOSITE PEAK FORCE CURVES (Physics vs Measured)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='F_physics_peak', y='F_measured_peak', hue='speed_kmph', palette='viridis', alpha=0.8)
lims = [min(df['F_physics_peak'].min(), df['F_measured_peak'].min()) - 50,
        max(df['F_physics_peak'].max(), df['F_measured_peak'].max()) + 50]
plt.plot(lims, lims, color='red', linestyle='--', label='Perfect Physical Match')
plt.title("Peak Force: Analytical Physics vs. Ground Truth 'Measured'")
plt.xlabel("Physics-Predicted Peak Force (N)")
plt.ylabel("Measured Peak Force (N)")
plt.legend()
plt.show()

print("EDA Complete. Key takeaways to check in plots:")
print("1. Anthropometrics check out (stiffness and leg lengths scale with physical build).")
print("2. The correlation heatmap reveals strong couplings between weight/speed and harmonic metrics.")
print("3. Residual plots for b2 and b3 show clear downward trends against BMI, confirming the unmodeled soft-tissue dampening effect waiting for the ML layer to crack.")

### 1. INVERTED-PENDULUM + LEG SPRING-DAMPER PHYSICS  (per-subject parameters)

In [ ]:




def leg_length_m(height_cm, sex):
    """Leg length as a fraction of height (Winter anthropometric tables)."""
    ratio = 0.535 if sex == "M" else 0.530
    return ratio * (height_cm / 100.0)


def leg_stiffness(weight_kg, L, age, sex):
    """Vertical leg stiffness (N/m), ~10-15 kN/m for a typical adult
    (Farley & Morgenroth, 1999), declining ~0.4%/yr after age 30."""
    base_k = 16.0 * weight_kg * G / L
    age_factor = 1 - max(0.0, (age - 30)) * 0.004
    sex_factor = 1.04 if sex == "M" else 1.0
    return base_k * age_factor * sex_factor


def damping_ratio(age):
    """Leg damping ratio zeta (typical human walking: 0.2-0.4), mildly rises
    with age (increased co-contraction / joint stiffening)."""
    return np.clip(0.26 + 0.0012 * (age - 30), 0.20, 0.42)


def froude_number(speed_kmph, L):
    """Dimensionless walking speed, Fr = v^2/(g*L) (Alexander, 1976). This is
    THE quantity that governs inverted-pendulum-like gait dynamics."""
    v = speed_kmph * 1000.0 / 3600.0
    return v ** 2 / (G * L)


In [ ]:
print(df.columns.tolist())

### 2. FOURIER-SERIES GRF MODEL

In [ ]:


K_REF, ZETA_REF = 12000.0, 0.28


def fourier_coeffs_physics(height_cm, weight_kg, age, sex, speed_kmph):
    """Physics-derived Fourier sine-series coefficients [b1,b2,b3,b4] (N) for
    the vertical GRF curve over one stance phase. Continuous in speed."""
    m = weight_kg
    L = leg_length_m(height_cm, sex)
    k = leg_stiffness(m, L, age, sex)
    zeta = damping_ratio(age)
    Fr = froude_number(speed_kmph, L)
    W = m * G  # body weight (N)

    b1 = W * 1.19                                                     # fundamental hump level
    b3 = W * 0.88 * np.sqrt(np.clip(Fr, 0, None)) * (k / K_REF)        # pendulum unloading -> M-shape depth
    b2 = W * 0.06 * (k / K_REF) * (1 - 0.6 * (zeta - ZETA_REF))        # heel-strike/push-off asymmetry
    b4 = W * 0.02 * (ZETA_REF / max(zeta, 1e-6))                       # secondary damping-driven asymmetry

    coeffs = np.array([b1, b2, b3, b4])
    meta = dict(L=L, k=k, zeta=zeta, Fr=Fr, W=W)
    return coeffs, meta


def reconstruct_curve(coeffs, n_points=101, W=None):
    """Sine-series reconstruction of the stance-phase GRF curve. phi=0 and
    phi=pi (heel-strike / toe-off) are automatically ~0 N by construction."""
    b1, b2, b3, b4 = coeffs
    phi = np.linspace(0, np.pi, n_points)
    F = b1 * np.sin(phi) + b2 * np.sin(2 * phi) + b3 * np.sin(3 * phi) + b4 * np.sin(4 * phi)
    F = np.clip(F, 0.0, None)
    if W is not None:
        F = np.clip(F, None, 1.6 * W)  # physiological safety ceiling for walking
    return phi, F


def fit_fourier_coeffs(phi, F, n_harmonics=4):
    """Bonus utility: project an ARBITRARY measured force curve onto the same
    sine basis (least-squares / Fourier projection), e.g. for real force-
    plate data. Because sin(n*phi) are orthogonal on [0,pi], each coefficient
    is just b_n = (2/pi) * integral( F(phi) * sin(n*phi) ) dphi."""
    coeffs = []
    for n in range(1, n_harmonics + 1):
        integrand = F * np.sin(n * phi)
        b_n = (2.0 / np.pi) * np.trapz(integrand, phi)
        coeffs.append(b_n)
    return np.array(coeffs)


def peak_force(coeffs, W):
    _, F = reconstruct_curve(coeffs, W=W)
    return F.max()

##Synthesising DataSet(will be removed later)

In [ ]:
# =============================================================================
# 3. SYNTHESIZE DATASET (200 subjects)
#    "measured" coefficients = physics coefficients, PLUS a systematic
#    effect the analytical model misses (soft-tissue damping at higher BMI
#    smooths the impact/push-off asymmetry -- a real, documented effect NOT
#    encoded in the physics formulas above), PLUS biological + sensor noise.
#    This gives the ML model genuine learnable signal to correct, mirroring
#    why physics-informed residual learning is useful in practice.
# =============================================================================

N_SAMPLES = 200
MEASURED_SPEEDS_KMPH = np.array([1, 2, 3, 4, 5, 6])

rows = []
for _ in range(N_SAMPLES):
    sex = RNG.choice(["M", "F"])
    if sex == "M":
        height = np.clip(RNG.normal(173, 7), 145, 200)
        weight = np.clip(RNG.normal(75, 11), 40, 130)
    else:
        height = np.clip(RNG.normal(160, 6), 145, 200)
        weight = np.clip(RNG.normal(62, 10), 40, 130)
    age = RNG.uniform(18, 70)
    speed = RNG.choice(MEASURED_SPEEDS_KMPH)

    coeffs, meta = fourier_coeffs_physics(height, weight, age, sex, speed)
    bmi = weight / ((height / 100) ** 2)

    # systematic real-world effect the hand-built physics model misses:
    soft_tissue_factor = 1 - 0.18 * np.clip((bmi - 22) / 10, 0, 1.2)
    coeffs_true = coeffs.copy()
    coeffs_true[1] *= soft_tissue_factor  # b2 asymmetry softened at high BMI
    coeffs_true[2] *= soft_tissue_factor  # b3 dip depth softened at high BMI

    noise_scale = np.array([0.05, 0.30, 0.20, 0.35])  # b1 varies less between people than shape harmonics do
    meas_coeffs = coeffs_true * (1 + RNG.normal(0, noise_scale)) + RNG.normal(0, 8, size=4)
    meas_coeffs = np.clip(meas_coeffs, 0, None)

    rows.append([height, weight, age, sex, speed, *coeffs, *meas_coeffs,
                 meta["Fr"], meta["k"], meta["zeta"], meta["W"], bmi])

COLS = ["height_cm", "weight_kg", "age", "sex", "speed_kmph",
        "b1_phys", "b2_phys", "b3_phys", "b4_phys",
        "b1_meas", "b2_meas", "b3_meas", "b4_meas",
        "Fr", "k_leg", "zeta", "W", "bmi"]
df = pd.DataFrame(rows, columns=COLS)
df["sex_num"] = (df["sex"] == "M").astype(int)
df["F_physics_peak"] = [peak_force(df.loc[i, ["b1_phys", "b2_phys", "b3_phys", "b4_phys"]].values, df.loc[i, "W"])
                         for i in df.index]
df["F_measured_peak"] = [peak_force(df.loc[i, ["b1_meas", "b2_meas", "b3_meas", "b4_meas"]].values, df.loc[i, "W"])
                          for i in df.index]
for i in range(1, 5):
    df[f"res_b{i}"] = df[f"b{i}_meas"] - df[f"b{i}_phys"]

df.to_csv("synthetic_force_fourier_dataset.csv", index=False)
print(f"Dataset created: {len(df)} rows -> synthetic_force_fourier_dataset.csv")
print(df[["height_cm", "weight_kg", "age", "sex", "speed_kmph",
          "F_physics_peak", "F_measured_peak"]].head(8).round(2))


###PHYSICS-INFORMED RESIDUAL ML  (predicts a correction to EACH harmonic)

In [ ]:

FEATURES = ["height_cm", "weight_kg", "age", "sex_num", "speed_kmph", "Fr", "k_leg", "zeta",
            "b1_phys", "b2_phys", "b3_phys", "b4_phys"]
TARGETS = ["res_b1", "res_b2", "res_b3", "res_b4"]

X = df[FEATURES]
Y = df[TARGETS]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

model = MultiOutputRegressor(
    GradientBoostingRegressor(n_estimators=250, max_depth=3, learning_rate=0.05,
                               subsample=0.8, random_state=42)
)
model.fit(X_train, Y_train)
pred_residual = model.predict(X_test)

phys_test = X_test[["b1_phys", "b2_phys", "b3_phys", "b4_phys"]].values
corrected_test = phys_test + pred_residual
actual_meas_test = df.loc[X_test.index, ["b1_meas", "b2_meas", "b3_meas", "b4_meas"]].values
W_test = df.loc[X_test.index, "W"].values

print("\n=== Per-harmonic performance on held-out test set ===")
for i, name in enumerate(["b1 (level)", "b2 (asymmetry)", "b3 (pendulum dip)", "b4 (asymmetry-2)"]):
    r2_ml = r2_score(actual_meas_test[:, i], corrected_test[:, i])
    mae_ml = mean_absolute_error(actual_meas_test[:, i], corrected_test[:, i])
    r2_ph = r2_score(actual_meas_test[:, i], phys_test[:, i])
    mae_ph = mean_absolute_error(actual_meas_test[:, i], phys_test[:, i])
    print(f"  {name:20s}  ML: R2={r2_ml:6.3f} MAE={mae_ml:6.2f} N   |  physics-only: R2={r2_ph:6.3f} MAE={mae_ph:6.2f} N")

peak_meas = np.array([peak_force(actual_meas_test[i], W_test[i]) for i in range(len(W_test))])
peak_phys = np.array([peak_force(phys_test[i], W_test[i]) for i in range(len(W_test))])
peak_corr = np.array([peak_force(corrected_test[i], W_test[i]) for i in range(len(W_test))])

print("\n=== Derived peak-force performance (reconstructed from the curves) ===")
print(f"  physics-only  MAE={mean_absolute_error(peak_meas, peak_phys):.2f} N  R2={r2_score(peak_meas, peak_phys):.3f}")
print(f"  ML-corrected  MAE={mean_absolute_error(peak_meas, peak_corr):.2f} N  R2={r2_score(peak_meas, peak_corr):.3f}")



In [ ]:
# 5. PREDICTION FUNCTION — full stance-phase curve, any subject, any speed
# =============================================================================

def predict_force_curve(height_cm, weight_kg, age, sex, speed_kmph, n_points=101):
    """Physics + ML-corrected vertical GRF curve over stance phase, for ANY
    (continuous, not just 1..6 km/h) walking speed."""
    sex_num = 1 if sex.upper() == "M" else 0
    L = leg_length_m(height_cm, sex)
    k = leg_stiffness(weight_kg, L, age, sex)
    zeta = damping_ratio(age)
    Fr = froude_number(speed_kmph, L)
    W = weight_kg * G

    phys_coeffs, meta = fourier_coeffs_physics(height_cm, weight_kg, age, sex, speed_kmph)
    row = pd.DataFrame([[height_cm, weight_kg, age, sex_num, speed_kmph, Fr, k, zeta, *phys_coeffs]],
                        columns=FEATURES)
    correction = model.predict(row)[0]
    corrected_coeffs = np.clip(phys_coeffs + correction, 0, None)
    phi, F = reconstruct_curve(corrected_coeffs, n_points=n_points, W=W)
    return phi, F, phys_coeffs, corrected_coeffs, meta


###Plots

In [ ]:
## PLOTS
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# --- (a) Example M-shaped curves: physics vs ML-corrected vs synthetic "measured" ---
ax = axes[0, 0]
example_idx = X_test.index[:3]
for idx, color in zip(example_idx, ["tab:blue", "tab:orange", "tab:green"]):
    h, w, a, s, sp = df.loc[idx, ["height_cm", "weight_kg", "age", "sex", "speed_kmph"]]
    phi, F_corr, phys_c, corr_c, meta = predict_force_curve(h, w, a, s, sp)
    _, F_phys = reconstruct_curve(phys_c, W=meta["W"])
    _, F_meas = reconstruct_curve(df.loc[idx, ["b1_meas", "b2_meas", "b3_meas", "b4_meas"]].values, W=meta["W"])
    pct_stance = phi / np.pi * 100
    ax.plot(pct_stance, F_phys, "--", color=color, alpha=0.6, linewidth=1.5)
    ax.plot(pct_stance, F_corr, "-", color=color, linewidth=2.2,
             label=f"{s}, {a:.0f}y, {sp}km/h")
    ax.scatter(pct_stance[::10], F_meas[::10], color=color, s=14, alpha=0.7)
ax.set_xlabel("% of stance phase")
ax.set_ylabel("Vertical GRF (N)")
ax.set_title("Inverted-pendulum + Fourier GRF curves\n(dashed=physics only, solid=ML-corrected, dots=synthetic 'measured')")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- (b) Pendulum-unloading effect: dip-depth harmonic b3 vs Froude number ---
ax = axes[0, 1]
ax.scatter(df["Fr"], df["b3_meas"], s=14, alpha=0.4, color="gray", label="synthetic 'measured' b3")
fr_line = np.linspace(0, df["Fr"].max(), 100)
ax.plot(fr_line, 0.88 * np.sqrt(fr_line) * (12000 / K_REF), color="crimson", linewidth=2,
        label=r"physics: $b_3 \propto \sqrt{Fr}$")
ax.set_xlabel("Froude number  $Fr = v^2/(gL)$")
ax.set_ylabel("b3 — pendulum dip-depth harmonic (N)")
ax.set_title("Inverted-pendulum unloading effect\n(faster relative walking speed -> deeper M-shape dip)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- (c) Predicted vs actual peak force ---
ax = axes[0, 2]
lims = [min(peak_meas.min(), peak_corr.min()) - 20, max(peak_meas.max(), peak_corr.max()) + 20]
ax.scatter(peak_meas, peak_phys, alpha=0.5, s=20, color="gray", label="physics-only")
ax.scatter(peak_meas, peak_corr, alpha=0.6, s=20, color="teal", label="ML-corrected")
ax.plot(lims, lims, "r--", linewidth=1, label="perfect prediction")
ax.set_xlabel("Actual (synthetic 'measured') peak force (N)")
ax.set_ylabel("Predicted peak force (N)")
ax.set_title("Peak force: predicted vs actual (test set)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- (d) Feature importance for the pendulum-dip (b3) correction model ---
ax = axes[1, 0]
b3_estimator = model.estimators_[2]  # index 2 = res_b3
importances = pd.Series(b3_estimator.feature_importances_, index=FEATURES).sort_values()
importances.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Feature importance — ML correction to b3\n(inverted-pendulum dip harmonic)")
ax.set_xlabel("Importance")

# --- (e) Force curves across speed for one demo subject ---
ax = axes[1, 1]
demo = dict(height_cm=175, weight_kg=70, age=30, sex="M")
for sp in [1.5, 3, 4.5, 6, 7.2]:
    phi, F, *_ = predict_force_curve(speed_kmph=sp, **demo)
    ax.plot(phi / np.pi * 100, F, label=f"{sp} km/h")
ax.set_xlabel("% of stance phase")
ax.set_ylabel("Vertical GRF (N)")
ax.set_title("Continuous prediction across ARBITRARY speeds\n(not restricted to the 1-6 km/h measured grid)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- (f) Peak force vs speed, continuous, few demo subjects ---
ax = axes[1, 2]
speeds_fine = np.linspace(0.5, 7.5, 60)
sample_people = [
    dict(height_cm=175, weight_kg=70, age=25, sex="M", label="Male, 25y, 70kg"),
    dict(height_cm=160, weight_kg=55, age=25, sex="F", label="Female, 25y, 55kg"),
    dict(height_cm=175, weight_kg=70, age=65, sex="M", label="Male, 65y, 70kg"),
]
for p in sample_people:
    p = dict(p)
    label = p.pop("label")
    peaks = []
    for sp in speeds_fine:
        _, F, *_ = predict_force_curve(speed_kmph=sp, **p)
        peaks.append(F.max())
    ax.plot(speeds_fine, peaks, linewidth=2, label=label)
ax.scatter(df["speed_kmph"], df["F_measured_peak"], s=10, alpha=0.2, color="gray", label="training data")
ax.set_xlabel("Speed (km/h)")
ax.set_ylabel("Predicted peak force (N)")
ax.set_title("Peak force vs speed (derived from full curve)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("force_model_fourier_results.png", dpi=150, bbox_inches="tight")
print("\nPlots saved -> force_model_fourier_results.png")
plt.show()

In [ ]:
8. #INTERACTIVE PREDICTION — ask the user for their own inputs
#    (run this cell/script directly; it will prompt you at the terminal)
# =============================================================================

# accuracy figures reported to the user come straight from the held-out test
# set computed in section 4 above, so they reflect genuine out-of-sample error
PEAK_MAE = mean_absolute_error(peak_meas, peak_corr)
PEAK_RMSE = np.sqrt(mean_squared_error(peak_meas, peak_corr))
PEAK_R2 = r2_score(peak_meas, peak_corr)
PEAK_MAPE = np.mean(np.abs((peak_meas - peak_corr) / peak_meas)) * 100


def get_float_input(prompt, min_val=None, max_val=None):
    while True:
        raw = input(prompt).strip()
        try:
            val = float(raw)
        except ValueError:
            print("  -> please enter a number.")
            continue
        if min_val is not None and val < min_val:
            print(f"  -> please enter a value >= {min_val}")
            continue
        if max_val is not None and val > max_val:
            print(f"  -> please enter a value <= {max_val}")
            continue
        return val


def get_sex_input(prompt="Sex (M/F): "):
    while True:
        raw = input(prompt).strip().upper()
        if raw in ("M", "MALE"):
            return "M"
        if raw in ("F", "FEMALE"):
            return "F"
        print("  -> please enter M or F.")



def run_interactive_prediction():
    print("\n" + "=" * 70)
    print("PREDICT VERTICAL GROUND REACTION FORCE FOR A NEW SUBJECT")
    print("=" * 70)
    height = get_float_input("Height (cm)            : ", 100, 230)
    weight = get_float_input("Weight (kg)             : ", 30, 200)
    age = get_float_input("Age (years)             : ", 5, 100)
    sex = get_sex_input("Sex (M/F)               : ")
    speed = get_float_input("Walking speed (km/h)    : ", 0.1, 10)

    phi, F, phys_coeffs, corr_coeffs, meta = predict_force_curve(height, weight, age, sex, speed)
    W = meta["W"]
    peak = F.max()
    mid_stance = F[len(F) // 2]

    print("\n--- Prediction ---")
    print(f"  Peak vertical GRF     : {peak:7.1f} N   ({peak / W:.2f} x body weight)")
    print(f"  Mid-stance trough     : {mid_stance:7.1f} N   ({mid_stance / W:.2f} x body weight)")
    print(f"  Froude number         : {meta['Fr']:.3f}")
    print(f"  Leg stiffness         : {meta['k']:.0f} N/m")

    print(f"\n--- Model accuracy (held-out test set, n={len(X_test)} of {len(df)} subjects) ---")
    print(f"  Peak-force MAE        : {PEAK_MAE:.1f} N")
    print(f"  Peak-force RMSE       : {PEAK_RMSE:.1f} N")
    print(f"  Peak-force R^2        : {PEAK_R2:.3f}")
    print(f"  Peak-force MAPE       : {PEAK_MAPE:.1f} %")
    print(f"  -> expect this prediction to be accurate to roughly +/-{PEAK_MAE:.0f} N on average "
          f"(about +/-{PEAK_MAE / W * 100:.0f}% of this subject's body weight).")

    plt.figure(figsize=(7, 5))
    pct_stance = phi / np.pi * 100
    plt.plot(pct_stance, F, linewidth=2.5, color="teal", label="ML-corrected prediction")
    plt.fill_between(pct_stance, np.clip(F - PEAK_MAE, 0, None), F + PEAK_MAE,
                      color="teal", alpha=0.15, label=f"+/- {PEAK_MAE:.0f} N (test-set MAE)")
    plt.xlabel("% of stance phase")
    plt.ylabel("Vertical GRF (N)")
    plt.title(f"Predicted GRF curve — {sex}, {age:.0f}y, {height:.0f}cm, {weight:.0f}kg, {speed:.1f}km/h")
    plt.legend(fontsize=9)

    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("user_prediction_curve.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\nCurve plot saved -> user_prediction_curve.png")


if __name__ == "__main__":
    run_interactive_prediction()



In [ ]:
# ==========================================================
# MODEL COMPARISON + HYPERPARAMETER TUNING
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

from sklearn.multioutput import MultiOutputRegressor

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.model_selection import RandomizedSearchCV

# ==========================================================
# MODELS
# ==========================================================

models = {

    "Linear Regression":
        MultiOutputRegressor(
            LinearRegression()
        ),

    "Ridge":
        MultiOutputRegressor(
            Ridge(alpha=1.0)
        ),

    "Decision Tree":
        MultiOutputRegressor(
            DecisionTreeRegressor(
                random_state=42
            )
        ),

    "Random Forest":
        MultiOutputRegressor(
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        ),

    "Extra Trees":
        MultiOutputRegressor(
            ExtraTreesRegressor(
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            )
        ),

    "Gradient Boosting":
        MultiOutputRegressor(
            GradientBoostingRegressor(
                random_state=42
            )
        ),

    "KNN":
        MultiOutputRegressor(
            KNeighborsRegressor(
                n_neighbors=5
            )
        ),

    "SVR":
        MultiOutputRegressor(
            SVR()
        )
}

# ==========================================================
# MODEL COMPARISON
# ==========================================================

comparison_results = []

phys_test = X_test[
    ["b1_phys","b2_phys","b3_phys","b4_phys"]
].values

actual = df.loc[
    X_test.index,
    ["b1_meas","b2_meas","b3_meas","b4_meas"]
].values

for name, model in models.items():

    model.fit(X_train, Y_train)

    pred = model.predict(X_test)

    corrected = phys_test + pred

    r2 = r2_score(
        actual,
        corrected,
        multioutput="uniform_average"
    )

    mae = mean_absolute_error(
        actual,
        corrected
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            corrected
        )
    )

    comparison_results.append([

        name,
        r2,
        mae,
        rmse

    ])

comparison = pd.DataFrame(

    comparison_results,

    columns=[
        "Model",
        "R2",
        "MAE",
        "RMSE"
    ]

)

comparison = comparison.sort_values(
    by="R2",
    ascending=False
)

print("\n==============================")
print("MODEL COMPARISON")
print("==============================")
print(comparison)

# ==========================================================
# BEST MODEL
# ==========================================================

best_name = comparison.iloc[0]["Model"]

print("\nBest Model :", best_name)

# ==========================================================
# PARAMETER GRIDS
# ==========================================================

param_grids = {

    "Random Forest":{

        "estimator__n_estimators":[100,200,300,500],

        "estimator__max_depth":[None,5,10,20],

        "estimator__min_samples_split":[2,5,10],

        "estimator__min_samples_leaf":[1,2,4]
    },

    "Extra Trees":{

        "estimator__n_estimators":[100,200,300,500],

        "estimator__max_depth":[None,5,10,20],

        "estimator__min_samples_split":[2,5,10],

        "estimator__min_samples_leaf":[1,2,4]
    },

    "Gradient Boosting":{

        "estimator__n_estimators":[100,200,300,500],

        "estimator__learning_rate":[0.01,0.03,0.05,0.1],

        "estimator__max_depth":[2,3,4,5],

        "estimator__subsample":[0.6,0.8,1.0],

        "estimator__min_samples_split":[2,5,10],

        "estimator__min_samples_leaf":[1,2,4]
    },

    "Decision Tree":{

        "estimator__max_depth":[None,5,10,20],

        "estimator__min_samples_split":[2,5,10],

        "estimator__min_samples_leaf":[1,2,4]
    },

    "KNN":{

        "estimator__n_neighbors":[3,5,7,9],

        "estimator__weights":["uniform","distance"]
    },

    "SVR":{

        "estimator__C":[0.1,1,10,100],

        "estimator__gamma":["scale","auto"],

        "estimator__kernel":["rbf","poly"]
    },

    "Ridge":{

        "estimator__alpha":[0.01,0.1,1,10,100]
    }
}

# ==========================================================
# HYPERPARAMETER TUNING
# ==========================================================

if best_name != "Linear Regression":

    best_model = models[best_name]

    search = RandomizedSearchCV(

        estimator=best_model,

        param_distributions=param_grids[best_name],

        n_iter=20,

        cv=5,

        scoring="r2",

        random_state=42,

        n_jobs=-1

    )

    search.fit(X_train,Y_train)

    final_model = search.best_estimator_

    print("\nBest Parameters")
    print(search.best_params_)

else:

    final_model = models[best_name]

    final_model.fit(X_train,Y_train)

# ==========================================================
# FINAL EVALUATION
# ==========================================================

pred = final_model.predict(X_test)

corrected = phys_test + pred

print("\n==============================")
print("FINAL MODEL PERFORMANCE")
print("==============================")

print("Average R2   :",round(
    r2_score(actual,corrected,multioutput="uniform_average"),4))

print("Average MAE  :",round(
    mean_absolute_error(actual,corrected),4))

print("Average RMSE :",round(
    np.sqrt(mean_squared_error(actual,corrected)),4))

print("\nPer Output Performance\n")

targets=[
    "b1",
    "b2",
    "b3",
    "b4"
]

for i,target in enumerate(targets):

    print(f"{target}")

    print("R2   :",round(
        r2_score(actual[:,i],corrected[:,i]),4))

    print("MAE  :",round(
        mean_absolute_error(actual[:,i],corrected[:,i]),4))

    print("RMSE :",round(
        np.sqrt(
            mean_squared_error(
                actual[:,i],
                corrected[:,i]
            )
        ),4))

    print()

# ==========================================================
# SAVE COMPARISON TABLE
# ==========================================================

comparison.to_csv(
    "model_comparison_results.csv",
    index=False
)

print("Model comparison saved.")

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import make_scorer, r2_score
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor
import numpy as np

cv_model = MultiOutputRegressor(
    GradientBoostingRegressor(
        random_state=42,
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8
    )
)

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    cv_model,
    X,
    Y,
    cv=kf,
    scoring=make_scorer(r2_score, multioutput='uniform_average')
)

print("========== 5-Fold Cross Validation ==========")
print("Fold Scores :", np.round(scores,4))
print("Mean R²     :", round(scores.mean(),4))
print("Std Dev     :", round(scores.std(),4))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor

base_model = MultiOutputRegressor(
    GradientBoostingRegressor(random_state=42)
)

param_grid = {

    "estimator__n_estimators":[100,200,300,400],

    "estimator__learning_rate":[
        0.01,
        0.03,
        0.05,
        0.1
    ],

    "estimator__max_depth":[
        2,3,4,5
    ],

    "estimator__subsample":[
        0.6,
        0.8,
        1.0
    ],

    "estimator__min_samples_split":[
        2,4,6
    ],

    "estimator__min_samples_leaf":[
        1,2,4
    ]

}

random_search = RandomizedSearchCV(

    estimator=base_model,

    param_distributions=param_grid,

    n_iter=20,

    cv=5,

    scoring="r2",

    verbose=2,

    random_state=42,

    n_jobs=-1

)

random_search.fit(X_train,Y_train)

print("\n========== BEST PARAMETERS ==========")
print(random_search.best_params_)

print("\nBest CV Score :",random_search.best_score_)

best_model=random_search.best_estimator_

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

pred = best_model.predict(X_test)

print("Overall Test R² :", r2_score(Y_test,pred))

for i,col in enumerate(TARGETS):

    print("-"*40)

    print(col)

    print("R² :",round(r2_score(Y_test.iloc[:,i],pred[:,i]),4))

    print("MAE:",round(mean_absolute_error(Y_test.iloc[:,i],pred[:,i]),4))

In [ ]:
import joblib

joblib.dump(model, "model.pkl")